# **Libraries**

In [53]:
import pandas as pd
import numpy as np
import requests

from sqlalchemy import create_engine

import folium
import json

from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

from math import radians, sin, cos, sqrt, atan2

from geopy.distance import geodesic
import polyline

# **Data Collection**

In [54]:
DATABASE_URL = "sqlite:///C:/Users/Xavier/Documents/Git Clone/samling-ai/backend/samling.db"
engine = create_engine(DATABASE_URL)

In [55]:
latest_batch = pd.read_sql("""
SELECT MAX(forecast_batch_id) AS batch
FROM volume_predictions
""", engine).iloc[0]["batch"]

latest_batch

'batch_20260713_225957'

In [56]:
forecast_df = pd.read_sql(f"""
SELECT *
FROM volume_predictions
WHERE forecast_batch_id = '{latest_batch}'
""", engine)

forecast_df.head()

,id,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan,tps_id,priority_rank,prediction_status,model_version
0,151,2026-07-13 15:59:57,90.961467,batch_20260713_225957,Cipayung,21,1,CRITICAL,v1.0
1,152,2026-07-13 15:59:57,85.612723,batch_20260713_225957,Mampang Prapatan,85,2,WARNING,v1.0
2,153,2026-07-13 15:59:57,83.692628,batch_20260713_225957,Cakung,2,3,WARNING,v1.0
3,154,2026-07-13 15:59:57,81.647069,batch_20260713_225957,Kebayoran Baru,57,4,WARNING,v1.0
4,155,2026-07-13 15:59:57,81.558708,batch_20260713_225957,Tanah Abang,143,5,WARNING,v1.0


In [57]:
zones = pd.read_sql("""
SELECT
    id,
    kecamatan,
    latitude,
    longitude
FROM zones
""", engine)

forecast_df = forecast_df.merge(
    zones,
    left_on="tps_id",
    right_on="id"
)

forecast_df.head()

,id_x,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan_x,tps_id,priority_rank,prediction_status,model_version,id_y,kecamatan_y,latitude,longitude
0,151,2026-07-13 15:59:57,90.961467,batch_20260713_225957,Cipayung,21,1,CRITICAL,v1.0,21,Cipayung,-6.270319,106.913076
1,152,2026-07-13 15:59:57,85.612723,batch_20260713_225957,Mampang Prapatan,85,2,WARNING,v1.0,85,Mampang Prapatan,-6.297520,106.776265
2,153,2026-07-13 15:59:57,83.692628,batch_20260713_225957,Cakung,2,3,WARNING,v1.0,2,Cakung,-6.283724,106.909670
3,154,2026-07-13 15:59:57,81.647069,batch_20260713_225957,Kebayoran Baru,57,4,WARNING,v1.0,57,Kebayoran Baru,-6.262663,106.781850
4,155,2026-07-13 15:59:57,81.558708,batch_20260713_225957,Tanah Abang,143,5,WARNING,v1.0,143,Tanah Abang,-6.174176,106.839083


In [58]:
forecast_df = forecast_df[
    forecast_df["prediction_status"] != "NORMAL"
].copy()

forecast_df = forecast_df.sort_values("priority_rank")

forecast_df.reset_index(drop=True, inplace=True)

forecast_df.head()

,id_x,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan_x,tps_id,priority_rank,prediction_status,model_version,id_y,kecamatan_y,latitude,longitude
0,151,2026-07-13 15:59:57,90.961467,batch_20260713_225957,Cipayung,21,1,CRITICAL,v1.0,21,Cipayung,-6.270319,106.913076
1,152,2026-07-13 15:59:57,85.612723,batch_20260713_225957,Mampang Prapatan,85,2,WARNING,v1.0,85,Mampang Prapatan,-6.297520,106.776265
2,153,2026-07-13 15:59:57,83.692628,batch_20260713_225957,Cakung,2,3,WARNING,v1.0,2,Cakung,-6.283724,106.909670
3,154,2026-07-13 15:59:57,81.647069,batch_20260713_225957,Kebayoran Baru,57,4,WARNING,v1.0,57,Kebayoran Baru,-6.262663,106.781850
4,155,2026-07-13 15:59:57,81.558708,batch_20260713_225957,Tanah Abang,143,5,WARNING,v1.0,143,Tanah Abang,-6.174176,106.839083


In [59]:
drivers = pd.read_sql("""
SELECT
    users.id,
    users.name,
    users.username,
    users.whatsapp_number,
    users.status,
    users.role,
    users.coverage_area,

    fleets.id AS fleet_id,
    fleets.name AS fleet_name,
    fleets.category,
    fleets.type,
    fleets.capacity,
    fleets.total_units

FROM users
LEFT JOIN fleets
    ON users.fleet_id = fleets.id

WHERE users.role = 'driver'
""", engine)

drivers["truck_capacity_kg"] = (
    drivers["capacity"]
    .str.extract(r"(\d+)")
    .astype(int)
    * 1000
)

drivers.head()

,id,name,username,whatsapp_number,status,role,coverage_area,fleet_id,fleet_name,category,type,capacity,total_units,truck_capacity_kg
0,2,Budi Utomo,driver_budi,6281234567890,Offline,driver,Jakarta Pusat,6,Dump Truck Besar,Tengah,Dump Truck,8 Ton,557,8000
1,3,Joko Susilo,driver_joko,6281298765432,Offline,driver,Jakarta Barat,5,Arm Roll Besar,Tengah,Arm Roll,5 Ton,143,5000
2,4,Agus Saputra,driver_agus,6281311223344,Offline,driver,Jakarta Selatan,7,Truk Compactor RDF,Tengah,Compactor,5 Ton,148,5000
3,5,Herman Wijaya,driver_herman,6281355667788,Offline,driver,Jakarta Timur,4,Arm Roll Kecil,Tengah,Arm Roll,3 Ton,200,3000
4,6,Rudy Hermawan,driver_rudy,6281288990011,Offline,driver,Jakarta Utara,8,Truk Tronton,Tengah,Truk Tronton,10 Ton,25,10000


In [60]:
with open('../../../core/depots.json', 'r', encoding='utf-8') as file:
    DEPOTS = json.load(file)

print(DEPOTS)

{'Jakarta Pusat': {'latitude': -6.1844, 'longitude': 106.8302, 'name': 'Depot DLH Jakarta Pusat'}, 'Jakarta Utara': {'latitude': -6.1244, 'longitude': 106.8912, 'name': 'Depot DLH Jakarta Utara'}, 'Jakarta Barat': {'latitude': -6.1674, 'longitude': 106.7637, 'name': 'Depot DLH Jakarta Barat'}, 'Jakarta Selatan': {'latitude': -6.2701, 'longitude': 106.8077, 'name': 'Depot DLH Jakarta Selatan'}, 'Jakarta Timur': {'latitude': -6.225, 'longitude': 106.9004, 'name': 'Depot DLH Jakarta Timur'}, 'Kepulauan Seribu': {'latitude': -5.6124, 'longitude': 106.5612, 'name': 'Depot DLH Kepulauan Seribu'}}


In [61]:
AREA_MAPPING = {
    "Cakung": "Jakarta Timur",
    "Ciracas": "Jakarta Timur",
    "Cipayung": "Jakarta Timur",
    "Duren Sawit": "Jakarta Timur",
    "Jatinegara": "Jakarta Timur",
    "Kramat Jati": "Jakarta Timur",
    "Makasar": "Jakarta Timur",
    "Matraman": "Jakarta Timur",
    "Pasar Rebo": "Jakarta Timur",
    "Pulogadung": "Jakarta Timur",

    "Cengkareng": "Jakarta Barat",
    "Grogol Petamburan": "Jakarta Barat",
    "Kalideres": "Jakarta Barat",
    "Kebon Jeruk": "Jakarta Barat",
    "Kembangan": "Jakarta Barat",
    "Palmerah": "Jakarta Barat",
    "Tambora": "Jakarta Barat",
    "Taman Sari": "Jakarta Barat",

    "Cilandak": "Jakarta Selatan",
    "Jagakarsa": "Jakarta Selatan",
    "Kebayoran Baru": "Jakarta Selatan",
    "Kebayoran Lama": "Jakarta Selatan",
    "Mampang Prapatan": "Jakarta Selatan",
    "Pancoran": "Jakarta Selatan",
    "Pasar Minggu": "Jakarta Selatan",
    "Pesanggrahan": "Jakarta Selatan",
    "Setiabudi": "Jakarta Selatan",
    "Tebet": "Jakarta Selatan",

    "Cempaka Putih": "Jakarta Pusat",
    "Gambir": "Jakarta Pusat",
    "Johar Baru": "Jakarta Pusat",
    "Kemayoran": "Jakarta Pusat",
    "Menteng": "Jakarta Pusat",
    "Sawah Besar": "Jakarta Pusat",
    "Senen": "Jakarta Pusat",
    "Tanah Abang": "Jakarta Pusat",

    "Cilincing": "Jakarta Utara",
    "Kelapa Gading": "Jakarta Utara",
    "Koja": "Jakarta Utara",
    "Pademangan": "Jakarta Utara",
    "Penjaringan": "Jakarta Utara",
    "Tanjung Priok": "Jakarta Utara",

    "Kepulauan Seribu Selatan": "Kepulauan Seribu",
    "Kepulauan Seribu Utara": "Kepulauan Seribu",
}

In [62]:
forecast_df["coverage_area"] = forecast_df["kecamatan_x"].map(AREA_MAPPING)
forecast_df

,id_x,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan_x,tps_id,priority_rank,prediction_status,model_version,id_y,kecamatan_y,latitude,longitude,coverage_area
0,151,2026-07-13 15:59:57,90.961467,batch_20260713_225957,Cipayung,21,1,CRITICAL,v1.0,21,Cipayung,-6.270319,106.913076,Jakarta Timur
1,152,2026-07-13 15:59:57,85.612723,batch_20260713_225957,Mampang Prapatan,85,2,WARNING,v1.0,85,Mampang Prapatan,-6.297520,106.776265,Jakarta Selatan
2,153,2026-07-13 15:59:57,83.692628,batch_20260713_225957,Cakung,2,3,WARNING,v1.0,2,Cakung,-6.283724,106.909670,Jakarta Timur
3,154,2026-07-13 15:59:57,81.647069,batch_20260713_225957,Kebayoran Baru,57,4,WARNING,v1.0,57,Kebayoran Baru,-6.262663,106.781850,Jakarta Selatan
4,155,2026-07-13 15:59:57,81.558708,batch_20260713_225957,Tanah Abang,143,5,WARNING,v1.0,143,Tanah Abang,-6.174176,106.839083,Jakarta Pusat
5,156,2026-07-13 15:59:57,79.077662,batch_20260713_225957,Jagakarsa,43,6,WARNING,v1.0,43,Jagakarsa,-6.269502,106.781816,Jakarta Selatan
6,157,2026-07-13 15:59:57,78.550416,batch_20260713_225957,Tanjung Priok,144,7,WARNING,v1.0,144,Tanjung Priok,-6.160781,106.916872,Jakarta Utara
7,158,2026-07-13 15:59:57,78.492578,batch_20260713_225957,Penjaringan,110,8,WARNING,v1.0,110,Penjaringan,-6.111131,106.915067,Jakarta Utara
8,159,2026-07-13 15:59:57,77.964339,batch_20260713_225957,Kepulauan Seribu Selatan,120,9,WARNING,v1.0,120,Kepulauan Seribu Selatan,-5.869289,106.565349,Kepulauan Seribu
9,160,2026-07-13 15:59:57,77.683122,batch_20260713_225957,Cilandak,12,10,WARNING,v1.0,12,Cilandak,-6.236315,106.784265,Jakarta Selatan


In [63]:
forecast_df = forecast_df.merge(
    drivers,
    left_on="coverage_area",
    right_on="coverage_area",
    how="left"
)

forecast_df.head()

,id_x,created_at,predicted_volume_percentage,forecast_batch_id,kecamatan_x,tps_id,priority_rank,prediction_status,model_version,id_y,...,whatsapp_number,status,role,fleet_id,fleet_name,category,type,capacity,total_units,truck_capacity_kg
0,151,2026-07-13 15:59:57,90.961467,batch_20260713_225957,Cipayung,21,1,CRITICAL,v1.0,21,...,6281355667788,Offline,driver,4.0,Arm Roll Kecil,Tengah,Arm Roll,3 Ton,200.0,3000.0
1,152,2026-07-13 15:59:57,85.612723,batch_20260713_225957,Mampang Prapatan,85,2,WARNING,v1.0,85,...,6281311223344,Offline,driver,7.0,Truk Compactor RDF,Tengah,Compactor,5 Ton,148.0,5000.0
2,153,2026-07-13 15:59:57,83.692628,batch_20260713_225957,Cakung,2,3,WARNING,v1.0,2,...,6281355667788,Offline,driver,4.0,Arm Roll Kecil,Tengah,Arm Roll,3 Ton,200.0,3000.0
3,154,2026-07-13 15:59:57,81.647069,batch_20260713_225957,Kebayoran Baru,57,4,WARNING,v1.0,57,...,6281311223344,Offline,driver,7.0,Truk Compactor RDF,Tengah,Compactor,5 Ton,148.0,5000.0
4,155,2026-07-13 15:59:57,81.558708,batch_20260713_225957,Tanah Abang,143,5,WARNING,v1.0,143,...,6281234567890,Offline,driver,6.0,Dump Truck Besar,Tengah,Dump Truck,8 Ton,557.0,8000.0


# **Routing**

In [64]:
def build_distance_matrix(points):

    coordinates = ";".join(
        f"{lon},{lat}"
        for lat, lon in points
    )

    url = (
        f"http://router.project-osrm.org/table/v1/driving/"
        f"{coordinates}"
        "?annotations=distance"
    )

    response = requests.get(url).json()

    return response["distances"]


def get_route_geometry(points):

    coordinates = ";".join(
        f"{lon},{lat}"
        for lat, lon in points
    )

    url = (
        f"http://router.project-osrm.org/route/v1/driving/"
        f"{coordinates}"
        "?overview=full"
        "&geometries=polyline"
    )

    response = requests.get(url).json()

    geometry = response["routes"][0]["geometry"]

    return polyline.decode(geometry)


def solve_route(distance_matrix):

    manager = pywrapcp.RoutingIndexManager(
        len(distance_matrix),
        1,
        0      # depot index
    )

    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):

        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        return int(distance_matrix[from_node][to_node])

    transit_callback = routing.RegisterTransitCallback(
        distance_callback
    )

    routing.SetArcCostEvaluatorOfAllVehicles(
        transit_callback
    )

    params = pywrapcp.DefaultRoutingSearchParameters()

    params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )

    solution = routing.SolveWithParameters(params)

    if solution is None:
        return None

    index = routing.Start(0)

    order = []

    while not routing.IsEnd(index):

        order.append(
            manager.IndexToNode(index)
        )

        index = solution.Value(
            routing.NextVar(index)
        )

    order.append(0)

    return order

In [65]:
routes = []

for area in forecast_df["coverage_area"].unique():

    tps = forecast_df[
        forecast_df["coverage_area"] == area
    ].copy()

    if tps.empty:
        continue

    matched_driver = drivers[
        drivers["coverage_area"] == area
    ]

    if matched_driver.empty:
        print(f"Skipping {area}: no driver assigned.")
        continue

    driver = matched_driver.iloc[0]

    # -------------------------------
    # Build point list
    # -------------------------------
    depot = DEPOTS[area]

    points = [
        (
            depot["latitude"],
            depot["longitude"]
        )
    ]

    points += list(
        zip(
            tps["latitude"],
            tps["longitude"]
        )
    )

    # -------------------------------
    # Route optimization
    # -------------------------------
    distance_matrix = build_distance_matrix(points)

    route = solve_route(distance_matrix)

    route_result = []

    for idx in route:

        if idx == 0:

            route_result.append({
                "type": "Depot",
                "name": area
            })

        else:

            row = tps.iloc[idx - 1]

            route_result.append({

                "type": "TPS",
                "tps_id": row.tps_id,
                "kecamatan": row.kecamatan_x,
                "priority_rank": row.priority_rank,
                "prediction": row.predicted_volume_percentage,
                "latitude": row.latitude,
                "longitude": row.longitude
            })

    routes.append({
        "area": area,
        "driver": driver["name"],
        "route": route_result
    })

Skipping Kepulauan Seribu: no driver assigned.


In [66]:
print(forecast_df["coverage_area"].unique())
print(drivers["coverage_area"].unique())

['Jakarta Timur' 'Jakarta Selatan' 'Jakarta Pusat' 'Jakarta Utara'
 'Kepulauan Seribu' 'Jakarta Barat']
['Jakarta Pusat' 'Jakarta Barat' 'Jakarta Selatan' 'Jakarta Timur'
 'Jakarta Utara']


In [67]:
print(routes)

[{'area': 'Jakarta Timur', 'driver': 'Herman Wijaya', 'route': [{'type': 'Depot', 'name': 'Jakarta Timur'}, {'type': 'TPS', 'tps_id': np.int64(80), 'kecamatan': 'Kramat Jati', 'priority_rank': np.int64(17), 'prediction': np.float64(74.0864264839487), 'latitude': np.float64(-6.2193562), 'longitude': np.float64(106.9188304)}, {'type': 'TPS', 'tps_id': np.int64(45), 'kecamatan': 'Jatinegara', 'priority_rank': np.int64(19), 'prediction': np.float64(73.09085570478365), 'latitude': np.float64(-6.2278972), 'longitude': np.float64(106.9156703)}, {'type': 'TPS', 'tps_id': np.int64(21), 'kecamatan': 'Cipayung', 'priority_rank': np.int64(1), 'prediction': np.float64(90.96146740467722), 'latitude': np.float64(-6.2703188), 'longitude': np.float64(106.9130757)}, {'type': 'TPS', 'tps_id': np.int64(2), 'kecamatan': 'Cakung', 'priority_rank': np.int64(3), 'prediction': np.float64(83.69262817917236), 'latitude': np.float64(-6.2837243), 'longitude': np.float64(106.9096703)}, {'type': 'Depot', 'name': 'Ja

In [68]:
m = folium.Map(
    location=[-6.20, 106.82],
    zoom_start=11
)

for area_route in routes:

    area = area_route["area"]
    route = area_route["route"]

    depot = DEPOTS[area]

    # Depot marker
    folium.Marker(
        location=[
            depot["latitude"],
            depot["longitude"]
        ],
        popup=depot["name"],
        icon=folium.Icon(color="green", icon="home")
    ).add_to(m)

    coordinates = []

    for stop in route:

        if stop["type"] == "Depot":

            coordinates.append([
                depot["latitude"],
                depot["longitude"]
            ])

        else:

            coordinates.append([
                stop["latitude"],
                stop["longitude"]
            ])

            folium.Marker(
                location=[
                    stop["latitude"],
                    stop["longitude"]
                ],
                popup=f"""
                TPS : {stop['tps_id']}<br>
                Rank : {stop['priority_rank']}<br>
                Pred : {stop['prediction']:.1f}%
                """,
                icon=folium.Icon(color="red")
            ).add_to(m)

    road = get_route_geometry(coordinates)

    folium.PolyLine(
        road,
        color="blue",
        weight=5,
        opacity=0.9
    ).add_to(m)

m